# ⚙️ _setup — Bootstrap de configuración

Este notebook se ejecuta al inicio de cada lab con `%run ./_setup`.
Lee **`config.py`** y expone:

| Variable | Descripción |
|---|---|
| `CATALOG` | Catálogo UC |
| `WORK_SCHEMA` | Esquema donde creas tus tablas |
| `DATA_PATH` | Carpeta con los Parquets de CFE (54 tablas) |
| `EJERCICIO1_PATH` | Carpeta con `tbobras.parquet` (labs 01 y 02) |
| `DELTA_LAB_FILE` | Ruta al Parquet `tbobras.parquet` (lab 01) |
| `PROYECTOS_FILE` | Ruta al Parquet `tbproyectos.parquet` |
| `CHK_PATH` | Volumen para checkpoints de streaming |
| `tbl(name)` | Helper: nombre calificado `catalog.schema.table` |

In [0]:
import os

def _config_dir():
    try:
        nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        return "/Workspace" + os.path.dirname(nb)
    except Exception:
        return os.getcwd()

_dir = _config_dir()
_config_path = os.path.join(_dir, "config.py")
assert os.path.exists(_config_path), (
    f"No se encontró config.py en {_dir}. Asegúrate de que config.py esté junto a los notebooks."
)
exec(open(_config_path).read())

DATA_PATH = DATA_PATH.rstrip("/")

# --- esquema de trabajo ---
_user = spark.sql("SELECT current_user()").collect()[0][0]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{WORK_SCHEMA}`")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{WORK_SCHEMA}`")

# --- volumen para checkpoints de streaming ---
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{WORK_SCHEMA}`.chk")
CHK_PATH = f"/Volumes/{CATALOG}/{WORK_SCHEMA}/chk"

def tbl(name: str) -> str:
    """Nombre calificado de una tabla en tu esquema de trabajo."""
    return f"`{CATALOG}`.`{WORK_SCHEMA}`.`{name}`"

print("=" * 60)
print(f"  Usuario         : {_user}")
print(f"  Catálogo        : {CATALOG}")
print(f"  Esquema trabajo : {WORK_SCHEMA}")
print(f"  DATA_PATH       : {DATA_PATH}")
print(f"  EJERCICIO1_PATH : {EJERCICIO1_PATH}")
print(f"  DELTA_LAB_FILE  : {DELTA_LAB_FILE}")
print(f"  CHK_PATH        : {CHK_PATH}")
print("=" * 60)